# 🏭 Создание нового проекта компании

Этот ноутбук:
1. Создаёт структуру папок и файлов для новой компании
2. Копирует все шаблоны YAML, ноутбуков, Excel
3. Настраивает конфиги под выбранный стандарт (US GAAP / IFRS)
4. Подсказывает следующие шаги

**Запустите все ячейки последовательно.**


In [ ]:
import sys, pathlib

# Auto-detect project root
ROOT = pathlib.Path('.').resolve()
for _p in [ROOT] + list(ROOT.parents):
    if (_p / 'engine').is_dir():
        ROOT = _p; break
sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')


## 1. Параметры компании

Заполните параметры ниже и запустите ячейку.


In [ ]:
# ═══════════════════════════════════════════════════════════
# ЗАПОЛНИТЕ ПАРАМЕТРЫ КОМПАНИИ
# ═══════════════════════════════════════════════════════════

COMPANY_ID = 'new_company'           # короткий ID (латиница, без пробелов)
COMPANY_NAME = 'New Company Inc.'     # полное название
INDUSTRY = 'metals'                   # metals | steel | energy | consumer | tech | financial | other
CURRENCY = 'USD'                      # USD | EUR | RUB | CNY
STANDARD = 'IFRS'                     # US_GAAP | IFRS

# ═══════════════════════════════════════════════════════════

print(f'Компания:  {COMPANY_NAME} ({COMPANY_ID})')
print(f'Отрасль:   {INDUSTRY}')
print(f'Валюта:    {CURRENCY}')
print(f'Стандарт:  {STANDARD}')


## 2. Создание структуры проекта

Запустите ячейку — будет создана полная структура компании.


In [ ]:
from tools.init_company import init_company

company_dir = init_company(
    company_id=COMPANY_ID,
    name=COMPANY_NAME,
    industry=INDUSTRY,
    currency=CURRENCY,
    standard=STANDARD,
    force=False,  # True = перезаписать если существует
)


## 3. Проверка структуры


In [ ]:
from pathlib import Path
import yaml

company_dir = ROOT / 'companies' / COMPANY_ID

# Check all expected files
expected_configs = [
    'configs/project.yaml',
    'configs/excel_loader.yaml',
    'configs/accounting_conventions.yaml',
    'configs/stress_scenarios.yaml',
    'configs/forecast/macro_ecm.yaml',
]
expected_dirs = [
    'data/excel', 'data/macro', 'data/debt', 'data/operational',
    'notebooks', 'outputs/model', 'outputs/stress',
]

print(f'📁 {company_dir}')
print(f'\n  Configs:')
for f in expected_configs:
    exists = (company_dir / f).exists()
    print(f'    {"✅" if exists else "❌"} {f}')

print(f'\n  Directories:')
for d in expected_dirs:
    exists = (company_dir / d).is_dir()
    print(f'    {"✅" if exists else "❌"} {d}/')

nbs = list((company_dir / 'notebooks').glob('*.ipynb'))
print(f'\n  Notebooks: {len(nbs)} files')
for nb in sorted(nbs):
    print(f'    ✅ {nb.name}')

# Show key settings
with open(company_dir / 'configs' / 'project.yaml') as f:
    cfg = yaml.safe_load(f)
ac = cfg.get('accounting_conventions', {})
print(f'\n  Key settings:')
print(f'    is_income_sign:  {ac.get("is_income_sign", "NOT SET")}')
print(f'    da_in_cogs:      {ac.get("da_in_cogs", "NOT SET")}')
print(f'    debt.mode:       {cfg.get("model",{}).get("standard",{}).get("debt",{}).get("mode", "?")}')


## 4. Следующие шаги

Проект создан! Теперь:

### Шаг 1: Подготовьте данные
1. Скопируйте шаблон Excel: `templates/excel_templates/template_UNIFIED_All_Data.xlsx`
2. Переименуйте в: `companies/{company}/data/excel/{company}_unified_complete.xlsx`
3. Заполните листы: `history_is`, `history_bs`, `history_cf`, `debt_instruments`, `macro_factors`

### Шаг 2: Настройте конфиги
Откройте ноутбуки в `companies/{company}/notebooks/`:
- **99_Configure_YAML.ipynb** — настройка модели через виджеты
- **99_Configure_Excel_Loader.ipynb** — настройка загрузки данных

### Шаг 3: Загрузите данные
- **01_Data_Loading.ipynb** — загрузка Excel → DB

### Шаг 4: Запустите модель
- **00_Build_Model_Main.ipynb** — полный pipeline

### Шаг 5: Анализ
- **02_Test_Model_Module.ipynb** — детальный анализ
- **03_Stress_Testing.ipynb** — стресс-тесты
- **04_Rating.ipynb** — кредитный рейтинг
- **05_Covenants.ipynb** — ковенанты


In [ ]:
# Quick links to next notebooks
from IPython.display import display, HTML

nb_dir = f'companies/{COMPANY_ID}/notebooks'
display(HTML(f'''
<h3>Ноутбуки компании {COMPANY_NAME}:</h3>
<ol>
<li><a href="../{nb_dir}/99_Configure_YAML.ipynb">99_Configure_YAML</a> — настройка project.yaml</li>
<li><a href="../{nb_dir}/01_Data_Loading.ipynb">01_Data_Loading</a> — загрузка данных</li>
<li><a href="../{nb_dir}/00_Build_Model_Main.ipynb">00_Build_Model_Main</a> — построение модели</li>
<li><a href="../{nb_dir}/02_Test_Model_Module.ipynb">02_Test_Model_Module</a> — анализ результатов</li>
</ol>
'''))
